# Neural-symbolic fusion: neural ensemble + symbolic v3 (URFD)

Neural: the 3-seed neck-centred ST-GCN ensemble, out-of-fold on test, each fold's own models on its validation videos. Symbolic: v3 (`symbolic_v3.py`, paper rule Eq. 8-10 on the neck height in metres), fitted on each fold's training videos only. v3 gives hard labels; for the probability-based fusions it gets a calibrated probability: the fall rate of training frames for each (trigger, state) pair.

All test numbers are post-hoc (the URFD test folds were seen before). Open from `notebooks/` or the project root and **Run All**.

In [1]:
import os, sys
import numpy as np
PROJECT_ROOT = os.environ.get("PROJECT_ROOT") or ("." if os.path.isdir("splits") else "..")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "fall_detection_models", "code", "stgcn_urfd"))
import symbolic_v3 as S3
import fusion_v4_experiments as X
X.ROOT = os.path.abspath(PROJECT_ROOT)
X.NPY = os.path.join(X.ROOT, "processed_data", "cfg_d31bfed741ab", "urfd_npy")
X.KIN = os.path.join(X.ROOT, "processed_data", "kinect_height")
REP = os.path.join(X.ROOT, "reports", "symbolic_v4")
split, data = X.load_all()
NV = X.neural_probs(os.path.join(REP, "neural_val.csv"), data)
NT = X.neural_probs(os.path.join(REP, "neural_test_oof.csv"), data)
neural = lambda part, v: NV[v] if part == "val" else NT[v]

SYM, LR = {}, {}
for f in split["folds"]:
    k = f["fold"]
    lying = np.concatenate([np.nan_to_num(data[v]["h"], nan=2.0)[data[v]["l3"] == 1] for v in f["train"] if v.startswith("fall")])
    m, train_mcc = S3.fit([(data[v]["h"], data[v]["y"]) for v in f["train"]], lying)
    tr = [m.predict(data[v]["h"]) for v in f["train"]]; y = np.concatenate([data[v]["y"] for v in f["train"]])
    code = np.concatenate([o["trigger"] * 2 + o["state"] for o in tr])
    P = {c: (y[code == c].sum() + 1) / ((code == c).sum() + 2) for c in range(4)}
    print(f"fold {k}: tau_h={m.tau_h:.3f} m, tau_v={m.tau_v:.3f} m/s, tau_s={m.tau_s:.3f} m | train MCC {train_mcc:.3f} | P(fall | trigger, state) =",
          {f"{c // 2},{c % 2}": round(p, 3) for c, p in P.items()})
    for part in ("val", "test"):
        for v in f[part]:
            o = m.predict(data[v]["h"])
            o["p"] = np.array([P[c] for c in o["trigger"] * 2 + o["state"]])
            o["upright"] = (np.nan_to_num(data[v]["h"], nan=0.0) >= 0.8).astype(int)
            SYM[(k, part, v)] = o
    Xv = np.vstack([np.c_[X.logit(NV[v]), SYM[(k, "val", v)]["p"]] for v in f["val"]])
    LR[k] = X.fit_lr(Xv, np.concatenate([data[v]["y"] for v in f["val"]]))

fold 0: tau_h=0.750 m, tau_v=0.411 m/s, tau_s=0.451 m | train MCC 0.452 | P(fall | trigger, state) = {'0,0': np.float64(0.084), '0,1': np.float64(0.795), '1,0': np.float64(0.922), '1,1': np.float64(0.992)}


fold 1: tau_h=0.750 m, tau_v=0.411 m/s, tau_s=0.374 m | train MCC 0.523 | P(fall | trigger, state) = {'0,0': np.float64(0.071), '0,1': np.float64(0.99), '1,0': np.float64(0.745), '1,1': np.float64(0.992)}


In [2]:
def run(meth, k, part, v):
    pn, s = neural(part, v), SYM[(k, part, v)]; n = (pn >= 0.5).astype(int)
    if meth == "neural":   return n
    if meth == "v3":       return s["trigger"]
    if meth == "paper":    return ((n | s["trigger"]) & s["support80"]).astype(int)
    if meth == "or":       return (n | s["trigger"]).astype(int)
    if meth == "veto":     return (n & (1 - s["upright"])).astype(int)
    if meth == "veto+trigger": return ((n & (1 - s["upright"])) | s["trigger"]).astype(int)
    if meth == "veto+state":   return ((n & (1 - s["upright"])) | s["state"]).astype(int)
    if meth == "mean":     return (((pn + s["p"]) / 2) >= 0.5).astype(int)
    if meth == "stacking": return (X.lr_predict(LR[k], np.c_[X.logit(pn), s["p"]]) >= 0.5).astype(int)

names = {"neural": "Neural ensemble alone", "v3": "Symbolic v3 alone (trigger)", "paper": "Paper Eq. 11-13",
         "or": "OR (neural or trigger)", "veto": "Gated veto (neck >= 0.8 m)", "veto+trigger": "Veto + trigger rescue",
         "veto+state": "Veto + state rescue", "mean": "Mean probability", "stacking": "Stacking (fit on own validation)"}
FUSED = {}
print(f"{'':34s} {'VAL':>6s} | {'TEST MCC':>8s} {'F1':>6s} {'Prec':>6s} {'Rec':>6s} {'Spec':>6s} {'Acc':>6s}  falls  ADL  alarms/min | fold0  fold1")
for meth, title in names.items():
    r = {}
    for part in ("val", "test"):
        rows = []
        for f in split["folds"]:
            for v in f[part]:
                p = run(meth, f["fold"], part, v); rows.append((v, data[v]["y"], p))
                if part == "test": FUSED[(meth, v)] = p
        r[part] = X.metrics(rows)
    pf = [X.metrics([(v, data[v]["y"], FUSED[(meth, v)]) for v in f["test"]])["MCC"] for f in split["folds"]]
    t = r["test"]
    print(f"{title:34s} {r['val']['MCC']:6.3f} | {t['MCC']:8.3f} {t['F1']:6.3f} {t['Precision']:6.3f} {t['Recall']:6.3f} {t['Specificity']:6.3f} "
          f"{t['Accuracy']:6.3f}  {t['falls_detected']}/{t['falls']}  {t['adl_clean']}/{t['adl']}  {t['fa_per_min']:5.2f} | {pf[0]:.3f}  {pf[1]:.3f}")

                                      VAL | TEST MCC     F1   Prec    Rec   Spec    Acc  falls  ADL  alarms/min | fold0  fold1
Neural ensemble alone               0.678 |    0.547  0.619  0.570  0.678  0.909  0.874  28/30  23/40   4.43 | 0.543  0.592
Symbolic v3 alone (trigger)         0.214 |    0.402  0.405  0.735  0.280  0.982  0.876  30/30  23/40   5.03 | 0.454  0.350
Paper Eq. 11-13                     0.455 |    0.448  0.520  0.580  0.471  0.939  0.868  30/30  28/40   3.62 | 0.395  0.516
OR (neural or trigger)              0.581 |    0.559  0.628  0.549  0.735  0.892  0.869  30/30  16/40   8.46 | 0.524  0.624
Gated veto (neck >= 0.8 m)          0.632 |    0.623  0.671  0.746  0.609  0.963  0.910  28/30  26/40   6.84 | 0.665  0.582
Veto + trigger rescue               0.535 |    0.621  0.677  0.688  0.666  0.946  0.904  30/30  18/40  10.87 | 0.627  0.615
Veto + state rescue                 0.388 |    0.511  0.589  0.517  0.686  0.886  0.856  30/30  22/40   8.66 | 0.482  0.542
Mean 

In [3]:
for meth in ("neural", "paper", "veto", "veto+trigger"):
    m = X.metrics([(v, data[v]["y"], FUSED[(meth, v)]) for f in split["folds"] for v in f["test"]])
    print(f"{names[meth]:34s} TN {m['tn']:5d}  FP {m['fp']:5d} | FN {m['fn']:5d}  TP {m['tp']:5d}")
missed = {meth: [v for f in split["folds"] for v in f["test"] if v.startswith("fall") and not FUSED[(meth, v)][int(np.argmax(data[v]["y"])):].any()]
          for meth in ("neural", "veto", "veto+trigger")}
print("falls never detected:", missed)
np.savez(os.path.join(PROJECT_ROOT, "fusion_v3_test_predictions.npz"), **{f"{v}__{m}": p for (m, v), p in FUSED.items()})
print("saved fusion_v3_test_predictions.npz")

Neural ensemble alone              TN  9210  FP   923 | FN   581  TP  1222
Paper Eq. 11-13                    TN  9517  FP   616 | FN   954  TP   849
Gated veto (neck >= 0.8 m)         TN  9760  FP   373 | FN   705  TP  1098
Veto + trigger rescue              TN  9589  FP   544 | FN   602  TP  1201
falls never detected: {'neural': ['fall-03', 'fall-11'], 'veto': ['fall-03', 'fall-11'], 'veto+trigger': []}
saved fusion_v3_test_predictions.npz
